# Agent Types

A complete reference notebook covering the main ways AI agents are classified:

1. **By Autonomy Level**
2. **By Architecture**
3. **By Planning Strategy**
4. **By Workflow Structure**

Each section includes an explanation and a small runnable code example that illustrates the concept.


---
## 1. By Autonomy Level

How much the agent relies on internal state, planning, and learning rather than reacting directly to input.


### 1.1 Simple Reflex Agent

Acts only on the **current input**, with no memory and no planning. Basically a set of condition-action rules (`if/else`).


In [ ]:
class SimpleReflexAgent:
    """Reacts only to the current percept, no memory at all."""

    def act(self, percept: str) -> str:
        if percept == "obstacle":
            return "turn_left"
        elif percept == "goal":
            return "stop"
        else:
            return "move_forward"


agent = SimpleReflexAgent()
for percept in ["clear", "obstacle", "clear", "goal"]:
    print(f"Percept: {percept:10s} -> Action: {agent.act(percept)}")


### 1.2 Model-Based Reflex Agent

Keeps an **internal state** (a model of the world) that gets updated over time, and uses it plus the current percept to decide.


In [ ]:
class ModelBasedReflexAgent:
    """Maintains an internal model of the world (state) across steps."""

    def __init__(self):
        self.visited_obstacles = 0
        self.history = []

    def act(self, percept: str) -> str:
        self.history.append(percept)

        if percept == "obstacle":
            self.visited_obstacles += 1

        # Decision depends on state, not just the current percept
        if self.visited_obstacles >= 2:
            return "reroute"
        elif percept == "obstacle":
            return "turn_left"
        elif percept == "goal":
            return "stop"
        return "move_forward"


agent = ModelBasedReflexAgent()
for percept in ["clear", "obstacle", "clear", "obstacle", "clear"]:
    print(f"Percept: {percept:10s} -> Action: {agent.act(percept):12s} | obstacles seen: {agent.visited_obstacles}")


### 1.3 Goal-Based Agent

Plans a **sequence of actions** to reach an explicit goal, instead of just reacting.


In [ ]:
from collections import deque

def goal_based_search(grid, start, goal):
    """Simple BFS pathfinding: the agent plans a path toward its goal."""
    rows, cols = len(grid), len(grid[0])
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == goal:
            return path

        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if (0 <= nr < rows and 0 <= nc < cols
                    and grid[nr][nc] == 0
                    and (nr, nc) not in visited):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [(nr, nc)]))
    return None


grid = [
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 1, 1, 0],
]

plan = goal_based_search(grid, start=(0, 0), goal=(3, 3))
print("Planned path to goal:", plan)


### 1.4 Utility-Based Agent

Doesn't just find *a* path to the goal — it scores multiple options with a **utility function** and picks the best one (e.g., shortest, safest, cheapest).


In [ ]:
routes = [
    {"name": "Route A", "distance_km": 12, "risk": 0.1, "time_min": 20},
    {"name": "Route B", "distance_km": 8,  "risk": 0.6, "time_min": 15},
    {"name": "Route C", "distance_km": 10, "risk": 0.2, "time_min": 18},
]

def utility(route, w_distance=0.3, w_risk=0.5, w_time=0.2):
    # Lower is better for all three -> utility = negative weighted cost
    cost = (w_distance * route["distance_km"]
            + w_risk * route["risk"] * 50   # scale risk up so it matters
            + w_time * route["time_min"])
    return -cost


best_route = max(routes, key=utility)
for r in routes:
    print(f"{r['name']:8s} utility = {utility(r):6.2f}")

print("\nChosen route:", best_route["name"])


### 1.5 Learning Agent

Improves its behavior over time from **experience** (e.g., reinforcement learning with an epsilon-greedy policy).


In [ ]:
import random

class LearningAgent:
    """A minimal epsilon-greedy learning agent for a 3-armed bandit problem."""

    def __init__(self, n_actions=3, epsilon=0.2, lr=0.1):
        self.q_values = [0.0] * n_actions
        self.epsilon = epsilon
        self.lr = lr

    def choose_action(self):
        if random.random() < self.epsilon:
            return random.randrange(len(self.q_values))   # explore
        return max(range(len(self.q_values)), key=lambda a: self.q_values[a])  # exploit

    def learn(self, action, reward):
        self.q_values[action] += self.lr * (reward - self.q_values[action])


# Simulated environment: action 2 gives the best average reward
true_rewards = [0.2, 0.5, 0.8]

agent = LearningAgent()
random.seed(0)
for step in range(200):
    action = agent.choose_action()
    reward = random.gauss(true_rewards[action], 0.1)
    agent.learn(action, reward)

print("Learned Q-values:", [round(q, 2) for q in agent.q_values])
print("Best action learned:", agent.q_values.index(max(agent.q_values)))


---
## 2. By Architecture

How the agent(s) are structured — one agent doing everything, several specialized agents, or a hierarchy of agents.


### 2.1 Single Agent

One agent handles planning, tool calling, and response generation by itself.

This is the pattern used by `create_agent(...)` in LangChain — a single LLM-driven agent equipped with tools.


In [ ]:
# Conceptual sketch (no API key required to read this structure)

# from langchain.agents import create_agent
#
# agent = create_agent(model="groq:openai/gpt-oss-20b", tools=[search_tool, calculator_tool])
# response = agent.invoke({"messages": [{"role": "user", "content": "What is 25 * 4?"}]})

class SingleAgent:
    """One agent that plans, calls tools, and responds -- all in one place."""

    def __init__(self, tools):
        self.tools = tools

    def run(self, task: str):
        print(f"[SingleAgent] Received task: {task}")
        print("[SingleAgent] Planning...")
        print("[SingleAgent] Calling tool if needed...")
        print("[SingleAgent] Generating final response.")
        return f"Result for: {task}"


agent = SingleAgent(tools=["search", "calculator"])
print(agent.run("Find the population of Cairo"))


### 2.2 Multi-Agent Systems

Several **specialized** agents cooperate, each responsible for part of the task (e.g., a researcher agent + a writer agent).


In [ ]:
class ResearcherAgent:
    def run(self, topic: str) -> str:
        return f"Facts gathered about '{topic}'"


class WriterAgent:
    def run(self, research: str) -> str:
        return f"Article written using: {research}"


class MultiAgentSystem:
    def __init__(self):
        self.researcher = ResearcherAgent()
        self.writer = WriterAgent()

    def run(self, topic: str) -> str:
        research = self.researcher.run(topic)
        article = self.writer.run(research)
        return article


system = MultiAgentSystem()
print(system.run("LangGraph"))


### 2.3 Hierarchical Agents (Supervisor)

A **supervisor/orchestrator** agent routes tasks to specialized sub-agents and combines their results.


In [ ]:
class SupervisorAgent:
    def __init__(self, sub_agents: dict):
        self.sub_agents = sub_agents   # e.g. {"math": MathAgent(), "search": SearchAgent()}

    def route(self, task: str) -> str:
        if any(op in task for op in ["+", "-", "*", "/"]):
            return "math"
        return "search"

    def run(self, task: str) -> str:
        chosen = self.route(task)
        print(f"[Supervisor] Routing task '{task}' -> '{chosen}' agent")
        return self.sub_agents[chosen].run(task)


class MathAgent:
    def run(self, task: str) -> str:
        return f"[MathAgent] Computed result for: {task}"


class SearchAgent:
    def run(self, task: str) -> str:
        return f"[SearchAgent] Found info for: {task}"


supervisor = SupervisorAgent({"math": MathAgent(), "search": SearchAgent()})
print(supervisor.run("25 * 4"))
print(supervisor.run("capital of France"))


---
## 3. By Planning Strategy

How the agent decides what to do next -- one step at a time, all upfront, or with self-review.


### 3.1 ReAct Agent (Reasoning + Acting)

Loops through: **think -> act -> observe -> think again**, one step at a time.


In [ ]:
class ReActAgent:
    def __init__(self, max_steps=4):
        self.max_steps = max_steps

    def run(self, goal: str):
        state = "start"
        for step in range(1, self.max_steps + 1):
            thought = f"Step {step}: thinking about how to reach '{goal}' from state '{state}'"
            print(thought)

            action = f"act_{step}"
            print(f"  -> Action: {action}")

            observation = "closer_to_goal" if step < self.max_steps else "goal_reached"
            print(f"  -> Observation: {observation}")

            state = observation
            if observation == "goal_reached":
                break

        return state


agent = ReActAgent()
final_state = agent.run("summarize a document")
print("\nFinal state:", final_state)


### 3.2 Plan-and-Execute Agent

Builds a **full plan upfront**, then executes it step by step.


In [ ]:
class PlanAndExecuteAgent:
    def plan(self, goal: str) -> list:
        # In a real agent, an LLM would generate this list
        return [f"Step {i}: work toward '{goal}'" for i in range(1, 4)]

    def execute(self, plan: list):
        results = []
        for step in plan:
            print(f"Executing -> {step}")
            results.append(f"done: {step}")
        return results


agent = PlanAndExecuteAgent()
plan = agent.plan("launch a marketing campaign")
print("Plan:", plan, "\n")
results = agent.execute(plan)
print("\nResults:", results)


### 3.3 Reflection Agent

Generates an output, then **critiques and refines** it before finalizing.


In [ ]:
class ReflectionAgent:
    def generate(self, task: str) -> str:
        return f"draft answer for '{task}'"

    def critique(self, draft: str) -> str:
        if "draft" in draft:
            return "Too informal, needs more detail"
        return "Looks good"

    def refine(self, draft: str, feedback: str) -> str:
        return f"refined answer ({feedback}) based on: {draft}"

    def run(self, task: str) -> str:
        draft = self.generate(task)
        print("Draft:", draft)

        feedback = self.critique(draft)
        print("Critique:", feedback)

        final = self.refine(draft, feedback)
        print("Final:", final)
        return final


agent = ReflectionAgent()
agent.run("explain LangGraph in one paragraph")


---
## 4. By Workflow Structure

This is the distinction that LangGraph focuses on directly: a **fixed graph** of steps vs. a **dynamically decided** path.


### 4.1 Workflow-based (Graph-based)

Nodes and edges are defined **explicitly** in advance -- a deterministic flow. This is exactly the pattern used throughout this course (`add_node`, `add_edge`, `set_entry_point`).


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class SimpleState(TypedDict):
    count: int


def increment(state: SimpleState) -> dict:
    return {"count": state["count"] + 1}


def report(state: SimpleState) -> dict:
    print(f"Current count: {state['count']}")
    return {}


workflow = StateGraph(SimpleState)
workflow.add_node("increment", increment)
workflow.add_node("report", report)

workflow.add_edge(START, "increment")
workflow.add_edge("increment", "report")
workflow.add_edge("report", END)

graph_app = workflow.compile()
result = graph_app.invoke({"count": 0})
print("Final state:", result)


### 4.2 Agentic (Dynamic)

The LLM itself decides the next step at runtime (via **conditional edges**), instead of following one fixed path.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class AgenticState(TypedDict):
    task: str
    needs_tool: bool
    result: str


def decide_node(state: AgenticState) -> dict:
    # In a real agent, an LLM decides this dynamically based on the task
    needs_tool = "calculate" in state["task"].lower()
    return {"needs_tool": needs_tool}


def tool_node(state: AgenticState) -> dict:
    return {"result": f"Used a tool to solve: {state['task']}"}


def direct_answer_node(state: AgenticState) -> dict:
    return {"result": f"Answered directly: {state['task']}"}


def route(state: AgenticState) -> str:
    return "tool_node" if state["needs_tool"] else "direct_answer_node"


workflow = StateGraph(AgenticState)
workflow.add_node("decide_node", decide_node)
workflow.add_node("tool_node", tool_node)
workflow.add_node("direct_answer_node", direct_answer_node)

workflow.add_edge(START, "decide_node")
workflow.add_conditional_edges(
    "decide_node",
    route,
    {"tool_node": "tool_node", "direct_answer_node": "direct_answer_node"},
)
workflow.add_edge("tool_node", END)
workflow.add_edge("direct_answer_node", END)

agentic_app = workflow.compile()

print(agentic_app.invoke({"task": "calculate 12 * 8", "needs_tool": False, "result": ""}))
print(agentic_app.invoke({"task": "say hello",       "needs_tool": False, "result": ""}))


---
## Summary Table

| Category | Types |
|---|---|
| **Autonomy Level** | Simple Reflex, Model-Based Reflex, Goal-Based, Utility-Based, Learning |
| **Architecture** | Single Agent, Multi-Agent, Hierarchical (Supervisor) |
| **Planning Strategy** | ReAct, Plan-and-Execute, Reflection |
| **Workflow Structure** | Workflow-based (Graph), Agentic (Dynamic) |

These categories are not mutually exclusive -- a real-world system (like the ones built with LangGraph) usually combines several of them, e.g. a **Multi-Agent**, **Hierarchical**, **ReAct**-style system built as a **Graph-based** workflow with some **Agentic** dynamic routing.
